In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class KMeansClustering():
    def __init__(self,k=5,epochs=1000,tol=1e-4):
        self.k = k
        self.epochs = epochs
        self.tol = tol
    def calculate_distance(self,X):
        n = X.shape[0]
        clusters = len(self.centroids)
        distances = np.zeros((n,clusters))
        for i in range(n):
            for j in range(clusters):
                diff = X[i]-self.centroids[j]
                dist = np.sqrt(np.sum(diff**2))
                distances[i][j] = dist
        return distances
    def assign_clusters(self,X):
        dist = self.calculate_distance(X)
        return np.argmin(dist,axis=1)
    def update_centroids(self,X,new_labels):
        new_centroids = np.zeros((self.k,X.shape[1]))
        for i in range(self.k):
            points = X[new_labels==i]
            if len(points)>0:
                new_centroids[i] = points.mean(axis=0)
            else:
                new_centroids[i] = self.centroids[i]
        return new_centroids
    def calculate_inertia(self,X):
        dist = self.calculate_distance(X)
        min_dist = np.min(dist,axis=1)
        return np.sum(min_dist**2)
    def fit(self,X,n_iters=10):
        indices = np.random.choice(len(X),self.k,replace=False)
        self.centroids = X[indices]
        min_inertia = float('inf')
        best_labels = None
        best_inertia = None
        best_centroids = None
        for k in range(n_iters):
            for i in range(self.epochs):
                labels = self.assign_clusters(X)
                new_centroids = self.update_centroids(X,labels)
                old_centroids = self.centroids
                self.centroids = new_centroids
                if np.sqrt(np.sum((new_centroids-old_centroids)**2))<=self.tol:
                    print(f"Converged at iteration {i+1}")
                    break
            inertia = self.calculate_inertia(X)
            if not best_inertia or inertia<best_inertia:
                best_inertia = inertia
                best_centroids = self.centroids
                best_labels = labels
                print(f"found better centroids at iteration{k+1}")
        self.centroids = best_centroids
        return best_labels,self.centroids
    def predict(self,X):
        return self.assign_clusters(X)

In [ ]:
# Create dummy data
X_sample = np.random.rand(100, 2) 

# Run the model
kmeans = KMeansClustering(k=3)
labels, centers = kmeans.fit(X_sample)

# Check results
print(f"Centroids shape: {centers.shape}") # Should be (3, 2)
print(f"Labels shape: {labels.shape}")     # Should be (100,)

In [ ]:
def run_test():
    print("Generating synthetic data...")
    # 1. Create 3 distinct groups of data (Blobs)
    np.random.seed(42) # For reproducibility
    
    # Blob 1 (bottom left)
    c1 = np.random.normal(loc=[2, 2], scale=0.5, size=(50, 2))
    # Blob 2 (right)
    c2 = np.random.normal(loc=[8, 3], scale=0.5, size=(50, 2))
    # Blob 3 (top)
    c3 = np.random.normal(loc=[5, 8], scale=0.5, size=(50, 2))
    
    # Stack them into one dataset X
    X = np.vstack([c1, c2, c3])
    
    print(f"Data shape: {X.shape}") # Should be (150, 2)

    # 2. Initialize and Fit
    print("\nRunning K-Means...")
    kmeans = KMeansClustering(k=3, epochs=1000, tol=1e-6)
    labels, centroids = kmeans.fit(X)

    # 3. Predict on new data (Optional check)
    new_points = np.array([[2, 2], [8, 3]])
    predictions = kmeans.predict(new_points)
    print(f"\nPrediction test: Point [2,2] assigned to cluster {predictions[0]}")
    print(f"Prediction test: Point [8,3] assigned to cluster {predictions[1]}")

    # 4. Visualize
    print("\nPlotting results...")
    plt.figure(figsize=(8, 6))
    
    # Plot the data points, colored by their cluster label
    plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', label='Data Points')
    
    # Plot the centroids as big red X's
    plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='x', s=200, label='Centroids')
    
    plt.title("Custom K-Means Results")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
run_test()